# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

Let's explore the structure of the dataset and see what record sets, fields, and columns are available.

In [ ]:
# List all record sets using their @id
print("Available record sets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# For each record set, list its fields and columns by @id
for record_set in metadata.record_sets:
    print(f"\nRecord Set: {record_set.name} (@id: {record_set.id})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        if hasattr(field, 'column') and field.column:
            print(f"      Source column @id: {field.column.id}")

## 3. Data Extraction
Load data from one or more specific record sets into DataFrames for analysis.

Use the `@id` references obtained above.

In [ ]:
# Extract data from each record set
record_sets_ids = [r.id for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}.")
    else:
        print(f"No records found for {record_set_id}.")

# Example: show columns and preview for the first available record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in data from {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No data extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field from the first record set for demonstration. Make sure to update the field `@id` based on the actual IDs found above.

In [ ]:
# Pick the first record set and find a numeric field
import numpy as np
record_set_id = list(dataframes.keys())[0] if dataframes else None
if record_set_id:
    df = dataframes[record_set_id]
    # Find a numeric field (float or integer) from the record set metadata
    # The @id is required; here we demonstrate with the first numeric field found
    numeric_field_id = None
    group_field_id = None
    for field in [f for f in metadata.record_sets if f.id==record_set_id][0].fields:
        if field.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
            numeric_field_id = field.id
            break
    for field in [f for f in metadata.record_sets if f.id==record_set_id][0].fields:
        if field.data_type in ['schema:Text', 'Text', 'schema:Category']:
            group_field_id = field.id
            break
    print(f"Using numeric field '@id': {numeric_field_id}")
    print(f"Using group field '@id': {group_field_id}")

    # If the field names are the @id, use as is; else map @id to column
    if numeric_field_id in df.columns:
        field_column = numeric_field_id
    else:
        # Try using field names instead
        field_column = df.columns[0]
        print(f"Defaulting to field column: {field_column}")
        numeric_field_id = field_column

    try:
        df[field_column] = pd.to_numeric(df[field_column], errors='coerce')
        threshold = df[field_column].mean()

        filtered_df = df[df[field_column] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{field_column}_normalized"] = (filtered_df[field_column] - filtered_df[field_column].mean()) / filtered_df[field_column].std()
        print(f"Normalized {field_column} for filtered records:")
        print(filtered_df[[field_column, f"{field_column}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"EDA steps could not be performed: {e}")
else:
    print("No available record set with extracted data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use Matplotlib and Seaborn for a quick visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric or group field for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:

* Load a Croissant-annotated dataset via the `mlcroissant` library
* Inspect metadata, record sets, fields, and their `@id` references
* Load records from the dataset for analysis
* Perform basic exploratory data analysis and visualization using dynamically selected `@id` field references

This workflow can be adapted to any dataset following the Croissant schema.